# Lecture 4: Data Visualisation

Building on the pandas foundations from [Lecture 3](lecture_3.ipynb), this lecture focuses on turning tabular data into figures that inform decisions. In [Lecture 5](lecture_5.ipynb) we then apply these ideas to model diagnostics.

:::{admonition} Learning Objectives
:class: tip
After this lecture, you will be able to:
- Explain why visualisation goes beyond descriptive statistics
- Apply best practices for effective and honest data visualisation
- Create publication-quality plots with matplotlib and seaborn
- Choose the right plot type for your data and audience
- Use interactive visualisation libraries (plotly, altair)
- Work with file paths using pathlib
:::

## Table of Contents

- [Motivation](#motivation)
- [Best Practices](#best-practices)
- [Plotting Libraries](#plotting-libraries) — matplotlib, seaborn, plotly, altair
- [When and What to Plot](#when-and-what-to-plot)
- [SWE: The pathlib Library](#swe-the-pathlib-library)
- [Exercises](#exercises)

## Motivation

### Data scientists visualise all the time

Visualisation is not just for presentations — it is a core analytical tool:
- Exploratory data analysis (EDA)
- Model diagnostics (residual plots, learning curves)
- Communicating results to stakeholders
- Detecting data quality issues

### Data vis. is not just descriptive statistics

**Anscombe's Quartet** {cite}`anscombe1973graphs` demonstrates that datasets with identical summary statistics (mean = 7.5, std = 2, correlation = 0.8) can have wildly different distributions — a linear fit, a curve, an outlier-driven cluster, and a vertical strip all share the same numbers.

More than four decades later, Matejka and Fitzmaurice's **Datasaurus Dozen** {cite}`matejka2017same` sharpened Anscombe's point with a computational twist: they used simulated annealing to construct *thirteen* two-dimensional datasets that all share the same summary statistics to two decimal places (means, standard deviations, and Pearson correlation), yet visually depict a star, an X, a dinosaur, and other unmistakable shapes. The lesson is the same, only more forceful: **summary statistics can be identical while the data is qualitatively different**, so a plot is not optional.

$\rightarrow$ Always visualise your data before modelling.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from fun_ds.plotting import set_lecture_style

set_lecture_style()

df = sns.load_dataset("anscombe")

# Summary statistics per subset — nearly identical
summary = df.groupby("dataset").agg(
    mean_x=("x", "mean"),
    mean_y=("y", "mean"),
    std_x=("x", "std"),
    std_y=("y", "std"),
    corr=("x", lambda s: s.corr(df.loc[s.index, "y"])),
).round(2)
print(summary)

g = sns.lmplot(
    x="x", y="y", col="dataset", data=df,
    col_wrap=2, ci=None, height=3.2,
    scatter_kws={"s": 40, "alpha": 0.8},
)
g.set_titles("dataset = {col_name}")
plt.tight_layout()
plt.show()

## Best Practices

### Principles of effective visualisation

Edward Tufte's work {cite}`tufte2001visual` gives us a compact checklist:

1. **Show the data** — maximise the data-ink ratio (Tufte)
2. **Be honest** — no misleading axes, truncations, or cherry-picking
3. **Guide the viewer** — clear titles, labels, and annotations
4. **Reduce clutter** — remove chart junk, unnecessary gridlines, 3D effects
5. **Choose appropriate encodings** — position > length > angle > area > colour

:::{note}
**Why that ordering?** The ranking of visual encodings is not aesthetic preference but an *empirical* finding. Cleveland and McGill's landmark 1984 paper "Graphical Perception: Theory, Experimentation, and Application to the Development of Graphical Methods" {cite}`cleveland1984graphical` ran controlled psychophysical experiments in which subjects estimated numerical values from different graphical encodings. They ranked the accuracy of ten *elementary perceptual tasks*: **position on a common scale** was most accurate, followed by **position on non-aligned scales**, **length**, **angle/slope**, **area**, **volume**, and finally **colour hue / saturation / density**. This is the reason a well-designed dot plot beats a pie chart for comparing quantities: the eye reads position more accurately than angle or area. Every modern style guide (Wilke, Tufte, ggplot2) traces this ordering to Cleveland & McGill.
:::

:::{note}
Prefer colour-blind-safe palettes (e.g. `viridis`, `cividis`, or seaborn's `colorblind`). Around 8% of men and 0.5% of women have some form of colour vision deficiency, so red-green contrasts often fail. Perceptually uniform maps also print well in greyscale.
:::

### Common pitfalls

:::{warning}
- Truncated y-axes that exaggerate differences
- Dual y-axes that mislead about correlations
- Pie charts for more than 3-4 categories
- Rainbow colour maps that obscure patterns
- 3D plots that distort proportions
:::

## Plotting Libraries

The Python plotting ecosystem is best understood along two axes: **imperative vs declarative**, and **static vs interactive**. Imperative libraries (matplotlib, seaborn) build figures step-by-step by mutating axes objects. Declarative libraries express a figure as a *specification* — data + marks + encodings — and delegate the rendering. The declarative approach descends from Wilkinson's **Grammar of Graphics** {cite}`wilkinson2005grammar`, which formalised a chart as a composition of statistical transformations, geometric objects (marks), scales, guides, and coordinate systems. R's `ggplot2` was the first widely used implementation; today, **Altair** (Python), **Plotly Express** (Python), and **ggplot2** (R) all share this lineage. The main practical benefit is that a small vocabulary composes into a huge range of charts without the user re-writing plotting code from scratch.

Underneath **Altair** sits [**Vega-Lite**](https://vega.github.io/vega-lite/) {cite}`satyanarayan2017vegalite`, a JSON grammar of interactive graphics developed at the University of Washington and published at IEEE VIS 2017. Altair is a thin Python API that emits Vega-Lite specifications; the browser then renders and manages interactions. **Plotly Express** takes a similar declarative posture on top of `plotly.js` / D3.

| Library | Paradigm | Type | Best For |
|---------|---|---|---|
| **matplotlib** | Imperative | Static | Full control, publication figures |
| **seaborn** | Imperative (statistical) | Static | Statistical visualisations, quick EDA |
| **plotly / Plotly Express** | Declarative | Interactive | Dashboards, exploratory drill-down |
| **altair** (Vega-Lite) | Declarative | Interactive | Grammar of graphics, concise specs |

### matplotlib

The foundational Python plotting library {cite}`hunter2007matplotlib`. Verbose but fully customisable — nearly every other library ultimately draws on top of it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fun_ds.data import load_california_housing
from fun_ds.plotting import set_lecture_style

set_lecture_style()
df = load_california_housing()

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Top-left: histogram of the target
axes[0, 0].hist(df["MedHouseVal"], bins=50, edgecolor="black", alpha=0.8)
axes[0, 0].set_xlabel("Median House Value")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].set_title("Target distribution")

# Top-right: scatter of income vs value
axes[0, 1].scatter(df["MedInc"], df["MedHouseVal"], alpha=0.25, s=6)
axes[0, 1].set_xlabel("Median Income")
axes[0, 1].set_ylabel("Median House Value")
axes[0, 1].set_title("Income vs house value")

# Bottom-left: boxplot of value by HouseAge bins
age_bins = pd.cut(df["HouseAge"], bins=[0, 10, 20, 30, 40, 60])
grouped = [df.loc[age_bins == b, "MedHouseVal"].values for b in age_bins.cat.categories]
axes[1, 0].boxplot(grouped, labels=[str(b) for b in age_bins.cat.categories])
axes[1, 0].set_xlabel("HouseAge bin")
axes[1, 0].set_ylabel("Median House Value")
axes[1, 0].set_title("Value by house age")
axes[1, 0].tick_params(axis="x", rotation=30)

# Bottom-right: correlation heatmap
corr = df.corr(numeric_only=True)
im = axes[1, 1].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[1, 1].set_xticks(range(len(corr.columns)))
axes[1, 1].set_yticks(range(len(corr.columns)))
axes[1, 1].set_xticklabels(corr.columns, rotation=45, ha="right")
axes[1, 1].set_yticklabels(corr.columns)
axes[1, 1].set_title("Feature correlations")
fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

### seaborn

A high-level interface for statistical graphics built on matplotlib {cite}`waskom2021seaborn`. Its defaults follow good visual practice, so you get sensible plots with very little code.

The pairplot below is the fastest way to eyeball marginal distributions and pairwise relationships at the same time — look for skew on the diagonal, non-linear structure off-diagonal, and clipping at value boundaries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from fun_ds.data import load_california_housing

df = load_california_housing()

subset = df[["MedInc", "HouseAge", "AveRooms", "MedHouseVal"]].sample(1000, random_state=0)

sns.pairplot(
    subset,
    diag_kind="kde",
    plot_kws={"alpha": 0.4, "s": 12},
    corner=True,
)
plt.suptitle("California housing — pairwise relationships", y=1.02)
plt.show()

### plotly

Interactive plots that work in notebooks and dashboards. Hover, zoom, and pan are especially useful when a scatter cloud hides structure at a glance — the tooltips let you interrogate individual points without re-plotting.

In [ ]:
from fun_ds.data import load_california_housing

df = load_california_housing()

try:
    import plotly.express as px

    sample = df.sample(2000, random_state=0)
    fig = px.scatter(
        sample,
        x="MedInc",
        y="MedHouseVal",
        color="HouseAge",
        hover_data=["AveRooms", "Population"],
        title="California housing: income vs price (colour = house age)",
        color_continuous_scale="Viridis",
    )
    fig.update_layout(width=750, height=500)
    fig.show()
except ImportError:
    print("plotly is not installed. Run `pip install plotly` to enable this cell.")

### altair

Declarative statistical visualisation based on Vega-Lite. Its API is a direct implementation of the *Grammar of Graphics* {cite}`wilkinson2005grammar`: you describe *what* to show (data, marks, encodings) and the library decides *how* to render it.

In [ ]:
from fun_ds.data import load_california_housing

df = load_california_housing()

try:
    import altair as alt

    sample = df.sample(2000, random_state=0)
    chart = (
        alt.Chart(sample)
        .mark_circle(size=20, opacity=0.5)
        .encode(
            x=alt.X("MedInc:Q", title="Median income"),
            y=alt.Y("MedHouseVal:Q", title="Median house value"),
            color=alt.Color("HouseAge:Q", scale=alt.Scale(scheme="viridis")),
            tooltip=["MedInc", "MedHouseVal", "HouseAge", "AveRooms"],
        )
        .properties(width=550, height=380, title="Income vs house value (altair)")
    )
    chart
except ImportError:
    print("altair is not installed. Run `pip install altair` to enable this cell.")
    chart = None
chart

### A note on colour theory

Colour is the *least* accurate visual channel in the Cleveland–McGill ranking {cite}`cleveland1984graphical`, but often the most abused. Two properties matter for honest data visualisation:

- **Perceptual uniformity.** A colormap is *perceptually uniform* if equal steps in the data correspond to equal *perceived* steps in colour (measured in a device-independent colour space such as CAM02-UCS). Old default rainbow maps (`jet`, `hsv`) are notoriously non-uniform: they compress differences in the yellow/cyan bands and inflate them at the red/blue ends, so equal data steps look unequal and the map can invent visual structure that is not in the data. The `viridis`, `plasma`, `inferno`, and `magma` colormaps introduced as matplotlib defaults in 2015 by Nathaniel Smith and Stéfan van der Walt (SciPy 2015 talk, *"A Better Default Colormap for Matplotlib"*) are designed for perceptual uniformity and print correctly in greyscale.
- **Colour-vision deficiency.** Approximately **8% of men** and **0.5% of women** of Northern European descent have some form of colour-vision deficiency, most commonly red–green. Red-vs-green encodings (traffic-light palettes) are therefore unsafe for a general audience. The viridis family, `cividis`, and seaborn's `colorblind` palette all remain distinguishable under the common deficiency simulations.

The practical rule: **use perceptually uniform sequential maps for ordered data** (`viridis`, `magma`), **diverging maps for signed data around a natural midpoint** (`RdBu_r`, `coolwarm`), and **qualitative palettes for unordered categories** (`tab10`, seaborn `colorblind`).

### Plot for the audience

Not every plot serves the same purpose, and treating them the same is one of the most common mistakes in applied work. Tufte's principles {cite}`tufte2001visual` apply differently at different stages:

| Purpose | EDA plot | Communication plot |
|---|---|---|
| Audience | The analyst | Stakeholder / reader |
| Message | *Many* questions, unknown answers | *One* claim to defend |
| Density | High — many small multiples, minimal polish | Low — one figure, heavily annotated |
| Lifecycle | Throw-away | Reproducible, versioned |
| Time invested | Seconds to minutes | Hours to days |
| Style | Defaults are fine; letter every axis | Every element deliberate; titles, legends, callouts |

The failure mode in either direction is symmetric. An over-polished EDA plot wastes hours refining a figure that answers a question you have already moved past. An under-polished communication plot buries the message in noise the reader has to filter. Recognising which mode you are in — and switching gears deliberately — is a large part of professional practice.

## When and What to Plot

The right plot depends on the question you are asking and the stage of the workflow. During EDA you cast a wide net — many small plots, minimal polish — to build intuition. During modelling you zero in on diagnostics that stress-test assumptions. When communicating, you strip everything to one or two annotated figures that carry a single message. The table below is a starting menu, not a prescription.

| Stage | Plot Type | Purpose |
|-------|-----------|----------|
| EDA | Histograms, boxplots, pairplots | Understand distributions |
| EDA | Scatter matrices, heatmaps | Find relationships |
| Modelling | Learning curves | Diagnose over/underfitting |
| Modelling | Residual plots | Check model assumptions |
| Communication | Bar charts, line plots | Present results clearly |
| Communication | Annotated scatter | Highlight key findings |

## SWE: The pathlib Library

Use `pathlib.Path` instead of string manipulation for file paths:
- Cross-platform (Windows/macOS/Linux)
- Readable and composable
- Rich API (`.exists()`, `.glob()`, `.stem`, `.suffix`)

In [ ]:
from pathlib import Path

# Current working directory
cwd = Path.cwd()
print(f"cwd: {cwd}")

# Parent directory of this notebook (in a .py file you'd use Path(__file__).parent)
notebook_dir = cwd
print(f"notebook dir: {notebook_dir}")

# Build a nested path and create it if missing
data_dir = cwd / "_scratch" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)
print(f"created: {data_dir} (exists={data_dir.exists()})")

# Glob for Python files in the current directory
py_files = list(cwd.glob("*.py"))
print(f"found {len(py_files)} .py files in {cwd.name}")

# Compose an output path from an input path
input_path = Path("data/raw/sales_2024.csv")
output_path = input_path.with_suffix(".parquet")
print(f"input:  {input_path}")
print(f"output: {output_path}  (stem={input_path.stem}, suffix={input_path.suffix})")

## Exercises

:::{admonition} Exercise 4.1 — EDA composite figure
:class: exercise
Using the California housing dataset, build a single figure with three panels: (a) a histogram of `MedHouseVal`, (b) a scatter of `MedInc` vs `MedHouseVal` coloured by `HouseAge`, and (c) a boxplot of `MedHouseVal` grouped by `AveRooms` bins. Add clear titles and axis labels. Which panel most changes your intuition about the target?
:::

:::{admonition} Exercise 4.2 — Fix a chart junk plot
:class: exercise
Reproduce a deliberately bad plot (e.g. a 3D pie chart of `HouseAge` bins with a rainbow palette, gridlines, and a truncated y-axis) and then produce a cleaned-up version applying Tufte's principles {cite}`tufte2001visual`. In one paragraph, list every element you removed and why.
:::

:::{admonition} Exercise 4.3 — Interactive plotly dashboard
:class: exercise
Build a small plotly figure (or `plotly.subplots` layout) that lets a reader hover to see `Latitude`, `Longitude`, `MedInc`, and `MedHouseVal` for individual districts. Colour points by `MedHouseVal` on a colour-blind-safe scale. Comment briefly on any spatial pattern you can see.
:::

:::{admonition} Key Takeaways
:class: important
- Always visualise data before modelling — summary statistics can mislead {cite}`anscombe1973graphs,matejka2017same`.
- Follow Tufte's principles {cite}`tufte2001visual`: maximise data-ink, minimise chart junk.
- The Cleveland–McGill ranking {cite}`cleveland1984graphical` orders encodings by empirical accuracy: position > length > angle > area > colour.
- Choose the right library: matplotlib for control, seaborn for statistics, plotly/altair (Vega-Lite {cite}`satyanarayan2017vegalite`) for interactivity.
- Match the plot type to the question you are answering, and distinguish EDA plots from communication plots.
- Use `pathlib` for robust cross-platform file handling.
:::